# 02 — Improved Forecasters: Hurdle-Aware Joint Probabilistic GRU

This notebook trains and evaluates forecasters on the LOTO folds produced by
`01_build_loto_folds_hurdle_targets.ipynb`. It reuses the exact model
architectures from the original WCNC notebook as baselines (`persistence`,
`deterministic_gru`, `independent_probabilistic_gru`, `joint_probabilistic_gru`
— the paper's current proposed model) and adds one new model:

### `hurdle_joint_probabilistic_gru` (proposed improvement)

Same low-rank joint Gaussian head as the original `JointProbabilisticGRU`,
**plus**:
1. An auxiliary linear head that predicts, for every future step and every
   gNB, the probability that a usable path will exist
   (`P(valid_path_count > 0)`), trained with binary cross-entropy against the
   `Y_valid` mask that the original notebook already computes but never uses
   in the loss.
2. A masked/hurdle training objective for the power head: at (step, gNB)
   entries where the ground-truth link is **not** usable, the regression
   target is replaced by the model's own (gradient-detached) prediction
   before computing the Gaussian NLL. This removes the gradient signal that
   would otherwise come from trying to match an arbitrary `-220 dB` sentinel,
   without changing the low-rank covariance structure or requiring a
   different distribution family.

All five models are trained **independently on each of the 6 LOTO folds**
(and, as an ablation, on both the compact `summary` and full `full` LiDAR
feature sets for the two joint models), so every reported number is a
**mean ± std over 6 held-out test trajectories** instead of one.

### Evaluation protocol (applied identically to every model)
* `masked_mae_db` / `masked_rmse_db`: error computed **only** at
  (step, gNB) entries where the ground-truth link is actually usable — the
  fair number to report as "forecast accuracy in dB".
* `raw_mae_db` / `raw_rmse_db`: the original (sentinel-inclusive) metric,
  kept so the new results stay comparable to the numbers already in the
  paper draft.
* `serving_gnb_accuracy`: unchanged from the original notebook.
* For the hurdle model only: link-detection `accuracy` / `precision` /
  `recall` / `F1` for the auxiliary classifier head.

Outputs are written to the new folder `improved_experiment_v2/{models,results}`.


In [1]:
from pathlib import Path
import copy
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.distributions import Normal, Independent, LowRankMultivariateNormal

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_ROOT = Path.home() / "wcnc_project"
V2_DATA_DIR = PROJECT_ROOT / "improved_experiment_v2" / "data"
V2_RESULTS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "results"
V2_MODELS_DIR = PROJECT_ROOT / "improved_experiment_v2" / "models"
V2_MODELS_DIR.mkdir(parents=True, exist_ok=True)
V2_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_FILE = V2_RESULTS_DIR / "loto_fold_manifest.json"
if not MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Missing {MANIFEST_FILE}. Run 01_build_loto_folds_hurdle_targets.ipynb first."
    )
with open(MANIFEST_FILE) as f:
    fold_manifest = json.load(f)

NUM_FOLDS = fold_manifest["num_trajectories"]
GNB_NAMES = fold_manifest["gnb_names"]
print(f"Found {NUM_FOLDS} LOTO folds for gNBs: {GNB_NAMES}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Found 6 LOTO folds for gNBs: ['gNB1_B59', 'gNB2_Building_44', 'gNB3_Building_54', 'gNB4_Building_29_Community_Center', 'gNB5_Building_617']
Training device: cuda
GPU: NVIDIA GeForce RTX 5090 Laptop GPU


In [3]:
# ------------------------------------------------------------
# Model definitions
# ------------------------------------------------------------

class GRUEncoder(nn.Module):
    """Shared temporal encoder — unchanged from the original notebook."""

    def __init__(self, input_dim, hidden_dim=128, number_of_layers=2, dropout=0.10):
        super().__init__()
        effective_dropout = dropout if number_of_layers > 1 else 0.0
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim,
                           num_layers=number_of_layers, batch_first=True,
                           dropout=effective_dropout)
        self.projection = nn.Sequential(
            nn.LayerNorm(hidden_dim), nn.Linear(hidden_dim, hidden_dim), nn.GELU()
        )

    def forward(self, x):
        _, hidden = self.gru(x)
        return self.projection(hidden[-1])


class DeterministicGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def forward(self, x):
        out = self.output_head(self.encoder(x))
        return out.view(-1, self.horizon, self.number_of_gnbs)


class IndependentProbabilisticGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.mean_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.variance_head = nn.Linear(hidden_dim, horizon * number_of_gnbs)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        variance = F.softplus(self.variance_head(context)) + 1e-3
        return Independent(Normal(mean, torch.sqrt(variance)), 1)

    def nll(self, x, y):
        dist = self.distribution(x)
        return -dist.log_prob(y.reshape(y.shape[0], -1))


class JointProbabilisticGRU(nn.Module):
    """The paper's current proposed model — low-rank joint Gaussian, unchanged."""

    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs, covariance_rank=4):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_dimension = horizon * number_of_gnbs
        self.covariance_rank = covariance_rank
        self.mean_head = nn.Linear(hidden_dim, self.output_dimension)
        self.diagonal_head = nn.Linear(hidden_dim, self.output_dimension)
        self.factor_head = nn.Linear(hidden_dim, self.output_dimension * covariance_rank)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        diagonal = F.softplus(self.diagonal_head(context)) + 1e-3
        factor = self.factor_head(context).view(-1, self.output_dimension, self.covariance_rank)
        factor = factor / np.sqrt(self.output_dimension)
        return LowRankMultivariateNormal(loc=mean, cov_factor=factor, cov_diag=diagonal)

    def nll(self, x, y):
        dist = self.distribution(x)
        return -dist.log_prob(y.reshape(y.shape[0], -1))


class HurdleJointProbabilisticGRU(nn.Module):
    """
    Proposed improved agentic forecaster.

    Adds a per-(step, gNB) link-availability classifier trained jointly with
    the low-rank joint Gaussian power forecaster. During training, targets
    for (step, gNB) entries with no usable link are replaced by the model's
    own detached prediction before computing the Gaussian NLL, so the loss
    is not dominated by an arbitrary sentinel floor value. The classifier
    head also gives the agentic handover policy a direct P(link usable)
    signal to gate candidate gNBs (see notebook 03).
    """

    def __init__(self, input_dim, hidden_dim, number_of_layers, horizon, number_of_gnbs,
                 covariance_rank=4, aux_weight=0.3):
        super().__init__()
        self.encoder = GRUEncoder(input_dim, hidden_dim, number_of_layers)
        self.output_dimension = horizon * number_of_gnbs
        self.covariance_rank = covariance_rank
        self.aux_weight = aux_weight
        self.mean_head = nn.Linear(hidden_dim, self.output_dimension)
        self.diagonal_head = nn.Linear(hidden_dim, self.output_dimension)
        self.factor_head = nn.Linear(hidden_dim, self.output_dimension * covariance_rank)
        self.link_head = nn.Linear(hidden_dim, self.output_dimension)
        self.horizon, self.number_of_gnbs = horizon, number_of_gnbs

    def distribution(self, x):
        context = self.encoder(x)
        mean = self.mean_head(context)
        diagonal = F.softplus(self.diagonal_head(context)) + 1e-3
        factor = self.factor_head(context).view(-1, self.output_dimension, self.covariance_rank)
        factor = factor / np.sqrt(self.output_dimension)
        return LowRankMultivariateNormal(loc=mean, cov_factor=factor, cov_diag=diagonal)

    def link_logits(self, x):
        return self.link_head(self.encoder(x))

    def link_probabilities(self, x):
        return torch.sigmoid(self.link_logits(x))

    def loss(self, x, y, y_valid):
        dist = self.distribution(x)
        target = y.reshape(y.shape[0], -1)
        mask_flat = y_valid.reshape(y_valid.shape[0], -1)
        masked_target = mask_flat * target + (1.0 - mask_flat) * dist.mean.detach()
        power_nll = -dist.log_prob(masked_target)
        link_logits = self.link_head(self.encoder(x))
        link_bce = F.binary_cross_entropy_with_logits(link_logits, mask_flat, reduction="none").mean(dim=1)
        return power_nll + self.aux_weight * link_bce


In [4]:
# ------------------------------------------------------------
# Training hyperparameters — identical to the original notebook
# ------------------------------------------------------------

HIDDEN_DIMENSION = 128
NUMBER_OF_LAYERS = 2
COVARIANCE_RANK = 4
AUX_WEIGHT = 0.3
BATCH_SIZE = 64
EPOCHS = 120
PATIENCE = 18
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4


def train_one_model(model, model_name, kind, train_X, train_Y, train_Yv, val_X, val_Y, val_Yv, verbose=True):
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    train_loader = DataLoader(
        TensorDataset(torch.from_numpy(train_X), torch.from_numpy(train_Y), torch.from_numpy(train_Yv)),
        batch_size=BATCH_SIZE, shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(torch.from_numpy(val_X), torch.from_numpy(val_Y), torch.from_numpy(val_Yv)),
        batch_size=BATCH_SIZE, shuffle=False,
    )

    best_val, best_state, bad_epochs = np.inf, None, 0
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for batch_X, batch_Y, batch_Yv in train_loader:
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)
            batch_Yv = batch_Yv.to(device)
            optimizer.zero_grad(set_to_none=True)
            if kind == "deterministic":
                loss = F.smooth_l1_loss(model(batch_X), batch_Y)
            elif kind == "probabilistic":
                loss = model.nll(batch_X, batch_Y).mean()
            elif kind == "hurdle":
                loss = model.loss(batch_X, batch_Y, batch_Yv).mean()
            else:
                raise ValueError(kind)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch_X, batch_Y, batch_Yv in val_loader:
                batch_X = batch_X.to(device)
                batch_Y = batch_Y.to(device)
                batch_Yv = batch_Yv.to(device)
                if kind == "deterministic":
                    vloss = F.smooth_l1_loss(model(batch_X), batch_Y)
                elif kind == "probabilistic":
                    vloss = model.nll(batch_X, batch_Y).mean()
                elif kind == "hurdle":
                    vloss = model.loss(batch_X, batch_Y, batch_Yv).mean()
                val_losses.append(float(vloss.detach().cpu()))

        mean_train, mean_val = float(np.mean(train_losses)), float(np.mean(val_losses))
        history["train_loss"].append(mean_train)
        history["val_loss"].append(mean_val)

        if mean_val < best_val:
            best_val, best_state, bad_epochs = mean_val, copy.deepcopy(model.state_dict()), 0
        else:
            bad_epochs += 1

        if verbose and (epoch == 1 or epoch % 20 == 0 or epoch == EPOCHS):
            print(f"    {model_name:32s} | epoch {epoch:3d} | train {mean_train:.4f} | val {mean_val:.4f}")
        if bad_epochs >= PATIENCE:
            if verbose:
                print(f"    {model_name}: early stopping at epoch {epoch}")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history, best_val


In [5]:
# ------------------------------------------------------------
# Fair evaluation helpers
# ------------------------------------------------------------

def inverse_norm(values_normalized, mean, std):
    return values_normalized * std.reshape(1, 1, -1) + mean.reshape(1, 1, -1)


def masked_point_metrics(pred_db, target_db, valid_mask, true_best_gnb):
    """masked_* only scores (step, gNB) entries where the ground-truth link is usable
    -- the fair number. raw_* reproduces the original notebook's sentinel-inclusive metric."""
    valid_mask_bool = valid_mask.astype(bool)
    error_db = pred_db - target_db

    if valid_mask_bool.any():
        masked_mae = float(np.mean(np.abs(error_db[valid_mask_bool])))
        masked_rmse = float(np.sqrt(np.mean(error_db[valid_mask_bool] ** 2)))
    else:
        masked_mae, masked_rmse = float("nan"), float("nan")

    raw_mae = float(np.mean(np.abs(error_db)))
    raw_rmse = float(np.sqrt(np.mean(error_db ** 2)))

    predicted_best = np.argmax(pred_db, axis=2)
    valid_best = true_best_gnb >= 0
    serving_accuracy = (
        float(np.mean(predicted_best[valid_best] == true_best_gnb[valid_best]))
        if valid_best.any() else float("nan")
    )

    return {
        "masked_mae_db": masked_mae, "masked_rmse_db": masked_rmse,
        "raw_mae_db": raw_mae, "raw_rmse_db": raw_rmse,
        "serving_gnb_accuracy": serving_accuracy,
        "fraction_valid_targets": float(valid_mask_bool.mean()),
    }


def link_classifier_metrics(link_prob, y_valid):
    predicted = (link_prob >= 0.5).astype(np.float32)
    truth = y_valid.astype(np.float32)
    tp = float(np.sum((predicted == 1) & (truth == 1)))
    fp = float(np.sum((predicted == 1) & (truth == 0)))
    fn = float(np.sum((predicted == 0) & (truth == 1)))
    tn = float(np.sum((predicted == 0) & (truth == 0)))
    precision = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    recall = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else float("nan")
    accuracy = (tp + tn) / (tp + fp + fn + tn)
    return {"link_accuracy": accuracy, "link_precision": precision, "link_recall": recall, "link_f1": f1}


In [6]:
# ------------------------------------------------------------
# Per-fold training + evaluation driver
# ------------------------------------------------------------

def run_fold(fold_index, lidar_mode, models_to_run):
    fold_file = V2_DATA_DIR / f"fold_{fold_index}_{lidar_mode}_sequences.npz"
    with np.load(fold_file, allow_pickle=True) as d:
        train_X, train_Y, train_Yv = d["train_X"], d["train_Y"], d["train_Y_valid"]
        val_X, val_Y, val_Yv = d["val_X"], d["val_Y"], d["val_Y_valid"]
        test_X, test_Y, test_Yv = d["test_X"], d["test_Y"], d["test_Y_valid"]
        test_best = d["test_future_best_gnb"]
        feature_mean, feature_std = d["feature_mean"], d["feature_std"]
        target_mean, target_std = d["target_mean"], d["target_std"]
        gnb_names = [str(x) for x in d["gnb_names"]]
        horizon = int(d["forecast_horizon"])

    number_of_gnbs = len(gnb_names)
    input_dim = train_X.shape[-1]
    test_target_db = inverse_norm(test_Y, target_mean, target_std)

    metrics, saved_scores = {}, {}
    common = dict(input_dim=input_dim, hidden_dim=HIDDEN_DIMENSION,
                  number_of_layers=NUMBER_OF_LAYERS, horizon=horizon, number_of_gnbs=number_of_gnbs)

    if "persistence" in models_to_run:
        last_power_norm = test_X[:, -1, :number_of_gnbs]
        last_power_db = last_power_norm * feature_std[:number_of_gnbs] + feature_mean[:number_of_gnbs]
        pred_db = np.repeat(last_power_db[:, None, :], horizon, axis=1)
        metrics["persistence"] = masked_point_metrics(pred_db, test_target_db, test_Yv, test_best)
        saved_scores["persistence"] = {"mean_db": pred_db}

    if "deterministic" in models_to_run:
        model = DeterministicGRU(**common)
        model, _, _ = train_one_model(model, f"deterministic[{lidar_mode}] fold{fold_index}", "deterministic",
                                       train_X, train_Y, train_Yv, val_X, val_Y, val_Yv)
        model.eval()
        with torch.no_grad():
            pred = model(torch.from_numpy(test_X).to(device)).cpu().numpy()
        pred_db = inverse_norm(pred, target_mean, target_std)
        metrics["deterministic"] = masked_point_metrics(pred_db, test_target_db, test_Yv, test_best)
        saved_scores["deterministic"] = {"mean_db": pred_db}
        torch.save(model.state_dict(), V2_MODELS_DIR / f"deterministic_{lidar_mode}_fold{fold_index}.pt")

    if "independent" in models_to_run:
        model = IndependentProbabilisticGRU(**common)
        model, _, _ = train_one_model(model, f"independent[{lidar_mode}] fold{fold_index}", "probabilistic",
                                       train_X, train_Y, train_Yv, val_X, val_Y, val_Yv)
        model.eval()
        with torch.no_grad():
            dist = model.distribution(torch.from_numpy(test_X).to(device))
            mean = dist.mean.reshape(-1, horizon, number_of_gnbs).cpu().numpy()
        pred_db = inverse_norm(mean, target_mean, target_std)
        metrics["independent"] = masked_point_metrics(pred_db, test_target_db, test_Yv, test_best)
        saved_scores["independent"] = {"mean_db": pred_db}
        torch.save(model.state_dict(), V2_MODELS_DIR / f"independent_{lidar_mode}_fold{fold_index}.pt")

    if "joint" in models_to_run:
        model = JointProbabilisticGRU(covariance_rank=COVARIANCE_RANK, **common)
        model, _, _ = train_one_model(model, f"joint[{lidar_mode}] fold{fold_index}", "probabilistic",
                                       train_X, train_Y, train_Yv, val_X, val_Y, val_Yv)
        model.eval()
        with torch.no_grad():
            tx = torch.from_numpy(test_X).to(device)
            dist = model.distribution(tx)
            mean = dist.mean.reshape(-1, horizon, number_of_gnbs).cpu().numpy()
            samples = dist.rsample((300,)).permute(1, 0, 2).reshape(-1, 300, horizon, number_of_gnbs).cpu().numpy()
        pred_db = inverse_norm(mean, target_mean, target_std)
        std_db = samples.std(axis=1) * target_std.reshape(1, 1, -1)
        metrics["joint"] = masked_point_metrics(pred_db, test_target_db, test_Yv, test_best)
        saved_scores["joint"] = {"mean_db": pred_db, "std_db": std_db}
        torch.save(model.state_dict(), V2_MODELS_DIR / f"joint_{lidar_mode}_fold{fold_index}.pt")

    if "hurdle_joint" in models_to_run:
        model = HurdleJointProbabilisticGRU(covariance_rank=COVARIANCE_RANK, aux_weight=AUX_WEIGHT, **common)
        model, _, _ = train_one_model(model, f"hurdle_joint[{lidar_mode}] fold{fold_index}", "hurdle",
                                       train_X, train_Y, train_Yv, val_X, val_Y, val_Yv)
        model.eval()
        with torch.no_grad():
            tx = torch.from_numpy(test_X).to(device)
            dist = model.distribution(tx)
            mean = dist.mean.reshape(-1, horizon, number_of_gnbs).cpu().numpy()
            samples = dist.rsample((300,)).permute(1, 0, 2).reshape(-1, 300, horizon, number_of_gnbs).cpu().numpy()
            link_prob = model.link_probabilities(tx).reshape(-1, horizon, number_of_gnbs).cpu().numpy()
        pred_db = inverse_norm(mean, target_mean, target_std)
        std_db = samples.std(axis=1) * target_std.reshape(1, 1, -1)
        fold_metrics = masked_point_metrics(pred_db, test_target_db, test_Yv, test_best)
        fold_metrics.update(link_classifier_metrics(link_prob, test_Yv))
        metrics["hurdle_joint"] = fold_metrics
        saved_scores["hurdle_joint"] = {"mean_db": pred_db, "std_db": std_db, "link_prob": link_prob}
        torch.save(model.state_dict(), V2_MODELS_DIR / f"hurdle_joint_{lidar_mode}_fold{fold_index}.pt")

    return metrics, saved_scores, dict(
        test_target_db=test_target_db, test_Yv=test_Yv, test_best=test_best,
        gnb_names=gnb_names, horizon=horizon, number_of_gnbs=number_of_gnbs,
    )


In [7]:
# ------------------------------------------------------------
# Main comparison: all 5 models, compact ("summary") LiDAR features, 6 LOTO folds
# ------------------------------------------------------------

MAIN_MODELS = ["persistence", "deterministic", "independent", "joint", "hurdle_joint"]

per_fold_results = {}
per_fold_scores = {}
per_fold_context = {}

for fold_index in range(NUM_FOLDS):
    print(f"\n=== Fold {fold_index} (lidar_mode=summary) ===")
    metrics, scores, context = run_fold(fold_index, "summary", MAIN_MODELS)
    per_fold_results[fold_index] = metrics
    per_fold_scores[fold_index] = scores
    per_fold_context[fold_index] = context

    np.savez_compressed(
        V2_RESULTS_DIR / f"fold_{fold_index}_summary_predictions.npz",
        **{f"{name}_mean_db": s["mean_db"] for name, s in scores.items()},
        **{f"{name}_std_db": s["std_db"] for name, s in scores.items() if "std_db" in s},
        **{f"{name}_link_prob": s["link_prob"] for name, s in scores.items() if "link_prob" in s},
        test_target_db=context["test_target_db"], test_Yv=context["test_Yv"], test_best=context["test_best"],
        gnb_names=np.asarray(context["gnb_names"], dtype="U"),
    )



=== Fold 0 (lidar_mode=summary) ===
    deterministic[summary] fold0     | epoch   1 | train 0.3089 | val 0.2210
    deterministic[summary] fold0     | epoch  20 | train 0.1044 | val 0.1581
    deterministic[summary] fold0: early stopping at epoch 36
    independent[summary] fold0       | epoch   1 | train 36.0893 | val 31.6408
    independent[summary] fold0       | epoch  20 | train 9.5339 | val 17.4808
    independent[summary] fold0       | epoch  40 | train 3.2450 | val 18.4638
    independent[summary] fold0: early stopping at epoch 46
    joint[summary] fold0             | epoch   1 | train 35.6100 | val 31.1018
    joint[summary] fold0             | epoch  20 | train 9.3580 | val 16.0261
    joint[summary] fold0             | epoch  40 | train 3.2177 | val 16.3461
    joint[summary] fold0: early stopping at epoch 40
    hurdle_joint[summary] fold0      | epoch   1 | train 23.2918 | val 15.3436
    hurdle_joint[summary] fold0      | epoch  20 | train -21.4015 | val -13.7335
    hu

In [8]:
# ------------------------------------------------------------
# Aggregate mean +/- std across the 6 folds, plus paired significance tests
# ------------------------------------------------------------

from scipy import stats

def aggregate_metric(model_name, metric_name):
    values = np.array([per_fold_results[k][model_name][metric_name] for k in range(NUM_FOLDS)
                        if metric_name in per_fold_results[k][model_name]], dtype=np.float64)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return None
    return float(values.mean()), float(values.std(ddof=1)) if len(values) > 1 else 0.0, values


summary_rows = []
key_metrics = ["masked_mae_db", "masked_rmse_db", "raw_mae_db", "raw_rmse_db", "serving_gnb_accuracy"]

print(f"{'model':16s} " + " | ".join(f"{m:22s}" for m in key_metrics))
for model_name in MAIN_MODELS:
    row = {"model": model_name}
    line = f"{model_name:16s} "
    for metric_name in key_metrics:
        agg = aggregate_metric(model_name, metric_name)
        if agg is None:
            line += " | " + " " * 22
            continue
        mean_v, std_v, _ = agg
        row[metric_name + "_mean"] = mean_v
        row[metric_name + "_std"] = std_v
        line += f" | {mean_v:8.3f} +/- {std_v:8.3f} "
    summary_rows.append(row)
    print(line)

# Link-detection quality (hurdle_joint only)
link_metrics = ["link_accuracy", "link_precision", "link_recall", "link_f1"]
print("\nHurdle-joint auxiliary link-availability classifier (mean +/- std over folds):")
for metric_name in link_metrics:
    agg = aggregate_metric("hurdle_joint", metric_name)
    if agg is not None:
        print(f"  {metric_name:16s} {agg[0]:.4f} +/- {agg[1]:.4f}")

# Paired significance test: hurdle_joint vs the paper's current proposed model (joint),
# paired by fold, on the fair masked_mae_db metric.
hurdle_vals = np.array([per_fold_results[k]["hurdle_joint"]["masked_mae_db"] for k in range(NUM_FOLDS)])
joint_vals = np.array([per_fold_results[k]["joint"]["masked_mae_db"] for k in range(NUM_FOLDS)])
t_stat, p_value = stats.ttest_rel(joint_vals, hurdle_vals)
print(f"\nPaired t-test (masked_mae_db), joint vs hurdle_joint across {NUM_FOLDS} folds:")
print(f"  mean masked MAE  joint={joint_vals.mean():.3f} dB   hurdle_joint={hurdle_vals.mean():.3f} dB")
print(f"  t={t_stat:.3f}  p={p_value:.4f}  (paired, one row per held-out trajectory)")

with open(V2_RESULTS_DIR / "forecaster_loto_summary.json", "w", encoding="utf-8") as f:
    json.dump({
        "num_folds": NUM_FOLDS,
        "models": MAIN_MODELS,
        "per_fold_results": {str(k): v for k, v in per_fold_results.items()},
        "aggregate_rows": summary_rows,
        "significance_masked_mae_joint_vs_hurdle_joint": {
            "t_stat": float(t_stat), "p_value": float(p_value),
            "joint_mean": float(joint_vals.mean()), "hurdle_joint_mean": float(hurdle_vals.mean()),
        },
    }, f, indent=2)

print("\nSaved:", V2_RESULTS_DIR / "forecaster_loto_summary.json")


model            masked_mae_db          | masked_rmse_db         | raw_mae_db             | raw_rmse_db            | serving_gnb_accuracy  
persistence       |   18.486 +/-    0.389  |   33.169 +/-    0.559  |   19.930 +/-    0.406  |   37.479 +/-    0.505  |    0.781 +/-    0.015 
deterministic     |   14.809 +/-    1.565  |   23.621 +/-    2.465  |   17.674 +/-    0.646  |   27.594 +/-    0.891  |    0.821 +/-    0.018 
independent       |   15.888 +/-    1.322  |   23.889 +/-    1.323  |   18.732 +/-    1.254  |   27.606 +/-    1.165  |    0.815 +/-    0.018 
joint             |   16.444 +/-    0.984  |   23.925 +/-    1.013  |   19.728 +/-    1.000  |   28.121 +/-    0.974  |    0.811 +/-    0.017 
hurdle_joint      |    7.527 +/-    0.780  |   10.250 +/-    1.701  |   32.201 +/-    0.748  |   49.329 +/-    0.701  |    0.701 +/-    0.046 

Hurdle-joint auxiliary link-availability classifier (mean +/- std over folds):
  link_accuracy    0.8247 +/- 0.0358
  link_precision   0.8294 +/

## Ablation: compact vs full LiDAR feature block

The main comparison above uses the compact 7-D `lidar_summary` features by
default. This cell repeats training for just the two joint models
(`joint`, `hurdle_joint`) with the original 72-D raw LiDAR rays
(`lidar_mode="full"`) across all 6 folds, to check whether the dimensionality
reduction helps, hurts, or is neutral on this dataset size. Skip this cell if
you only need the headline comparison above — it roughly doubles the
notebook''s runtime.


In [9]:
ABLATION_MODELS = ["joint", "hurdle_joint"]
ablation_results = {"summary": {}, "full": {}}

for lidar_mode in ["summary", "full"]:
    for fold_index in range(NUM_FOLDS):
        if lidar_mode == "summary":
            # Already computed above -- reuse instead of retraining.
            ablation_results["summary"][fold_index] = {
                m: per_fold_results[fold_index][m] for m in ABLATION_MODELS
            }
            continue
        print(f"\n=== Ablation fold {fold_index} (lidar_mode=full) ===")
        metrics, _, _ = run_fold(fold_index, "full", ABLATION_MODELS)
        ablation_results["full"][fold_index] = metrics

print("\nLiDAR feature ablation (masked_mae_db, mean +/- std over folds):")
for model_name in ABLATION_MODELS:
    for lidar_mode in ["summary", "full"]:
        vals = np.array([ablation_results[lidar_mode][k][model_name]["masked_mae_db"] for k in range(NUM_FOLDS)])
        vals = vals[~np.isnan(vals)]
        print(f"  {model_name:14s} lidar_mode={lidar_mode:7s} {vals.mean():.3f} +/- {vals.std(ddof=1):.3f} dB")

with open(V2_RESULTS_DIR / "lidar_feature_ablation.json", "w", encoding="utf-8") as f:
    json.dump({
        lidar_mode: {str(k): v for k, v in results.items()}
        for lidar_mode, results in ablation_results.items()
    }, f, indent=2)

print("\nSaved:", V2_RESULTS_DIR / "lidar_feature_ablation.json")



=== Ablation fold 0 (lidar_mode=full) ===
    joint[full] fold0                | epoch   1 | train 36.2704 | val 31.4396
    joint[full] fold0                | epoch  20 | train 5.9143 | val 16.0026
    joint[full] fold0                | epoch  40 | train -0.1175 | val 21.8762
    joint[full] fold0: early stopping at epoch 40
    hurdle_joint[full] fold0         | epoch   1 | train 22.1688 | val 13.8560
    hurdle_joint[full] fold0         | epoch  20 | train -24.1135 | val -12.1646
    hurdle_joint[full] fold0: early stopping at epoch 31

=== Ablation fold 1 (lidar_mode=full) ===
    joint[full] fold1                | epoch   1 | train 36.2606 | val 32.0048
    joint[full] fold1                | epoch  20 | train 5.2791 | val 17.7547
    joint[full] fold1: early stopping at epoch 33
    hurdle_joint[full] fold1         | epoch   1 | train 23.2270 | val 14.9216
    hurdle_joint[full] fold1         | epoch  20 | train -24.1307 | val -21.4695
    hurdle_joint[full] fold1         | epoch